# Zadanie 3.1: jednoprzedziałowy surogat ODE (`ODE_1`)

## Wyprowadzenie z całki po przestrzeni

Całkujemy równanie bazowe PDE po obszarze $\Omega = [-L/2, L/2]$:
$$\partial_t u = D\,\partial_{xx} u + \rho u\!\left(1 - \frac{u}{K}\right) - \beta R(x)\,U(t)\,u.$$
Przy warunkach brzegowych Neumanna ($\partial_x u = 0$ na obu końcach) człon dyfuzyjny znika z całki:
$$\int_\Omega D\,\partial_{xx} u\,dx = D\big[\partial_x u\big]_{-L/2}^{L/2} = 0.$$
Oznaczając całkowitą masę guza $M(t) = \int_\Omega u(x, t)\,dx$, dostajemy równanie ścisłe:
$$\frac{dM}{dt} = \rho M - \frac{\rho}{K}\int_\Omega u(x, t)^2\,dx - \beta U(t)\int_\Omega R(x)\,u(x, t)\,dx.$$

Żeby zamknąć równanie względem samej masy $M(t)$, wprowadzamy dwa przybliżenia przestrzenne:
1. **Efektywna szerokość profilu** $W_{\text{eff}}$: $\int_\Omega u^2\,dx \approx M^2 / W_{\text{eff}}$, co daje efektywną pojemność $K_{\text{eff}} = K\,W_{\text{eff}}$.
2. **Stały ułamek guza w wiązce** $c \in [0, 1]$: $\int_\Omega R(x)\,u(x, t)\,dx \approx c\,M(t)$.

Otrzymujemy jednoprzedziałowy model **`ODE_1`**:
$$\frac{dM}{dt} = \rho M\!\left(1 - \frac{M}{K_{\text{eff}}}\right) - \beta c\,U(t)\,M.$$

Jako naturalny punkt wyjścia (przed ręcznym strojeniem w #20) bierzemy:
- $c = c_0 = \frac{\int R(x)u_0(x)\,dx}{\int u_0(x)\,dx}$ — początkowy ułamek masy guza w polu wiązki,
- $K_{\text{eff}}$ dobrane tak, aby dla guza bez leczenia model logistyczny odtwarzał wzrost $M(t)$ na horyzoncie $t \in [0, 10]$ (fala Fishera rośnie liniowo w czasie, więc jedna stała $K_{\text{eff}}$ jest kompromisem dla zadanego okna).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import NO_TREATMENT, Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.ode import ODE1Params, initial_beam_fraction, simulate_ode1
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)

# Odniesienie bez leczenia: szerokość efektywna rośnie w czasie z prędkością frontu 2*sqrt(D*rho)=2.
u_free = simulate(Params(), NO_TREATMENT, x, t, method="cn")
M_free = tumor_mass(u_free, x)
print(f"M(0) = {M0:.2f}, bez leczenia M(10) = {M_free[-1]:.2f}")

# Wybór ręczny bazowego K_eff dla okna [0, 10] (w #20 dobór szczegółowy).
K_EFF_BASE = 45.0

scenarios = {
    "pole pokrywa guz": Params(beta=1.5, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}

## Porównanie `ODE_1` (z $c = c_0$) z pełnym PDE w trzech scenariuszach

In [ ]:
rows = []
curves = {}
for name, p in scenarios.items():
    u_pde = simulate(p, schedule, x, t, method="cn")
    M_pde = tumor_mass(u_pde, x)
    c0 = initial_beam_fraction(x, u0, p)
    ode_p = ODE1Params(rho=p.rho, K_eff=K_EFF_BASE, beta=p.beta, c=c0)
    M_ode = simulate_ode1(M0, ode_p, schedule, t, dose="pulse")
    curves[name] = (M_pde, M_ode, c0)

    mmin_p, tmin_p = min_mass(t, M_pde)
    mmin_o, tmin_o = min_mass(t, M_ode)
    treg_p = time_to_regrowth(t, M_pde)
    treg_o = time_to_regrowth(t, M_ode)
    rel_l2 = float(np.linalg.norm(M_ode - M_pde) / np.linalg.norm(M_pde))
    rows.append({
        "scenariusz": name,
        "c0": round(c0, 3),
        "K_eff": K_EFF_BASE,
        "M_min_PDE": round(mmin_p, 3),
        "M_min_ODE1": round(mmin_o, 3),
        "M_end_PDE": round(float(M_pde[-1]), 2),
        "M_end_ODE1": round(float(M_ode[-1]), 2),
        "t_odrostu_PDE": round(treg_p, 2) if np.isfinite(treg_p) else np.nan,
        "t_odrostu_ODE1": round(treg_o, 2) if np.isfinite(treg_o) else np.nan,
        "blad_L2_wzgl": round(rel_l2, 3),
    })

df = pd.DataFrame(rows)
df.to_csv(TABLES / "zad3_01_ode1.csv", index=False)
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (name, (M_pde, M_ode, c0)) in zip(axes, curves.items()):
    ax.plot(t, M_pde, color="black", lw=2.0, label="PDE (CN)")
    ax.plot(t, M_ode, color="tab:blue", ls="--", lw=2.0, label=f"ODE_1 ($c_0={c0:.2f}$)")
    for ti in schedule.times:
        ax.axvline(ti, color="tab:red", alpha=0.14, lw=0.8)
    ax.axhline(M0, color="gray", ls=":", lw=0.9)
    ax.set_title(name)
    ax.set_xlabel("czas $t$")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("masa guza $M(t)$")
for ax in axes: ax.legend(frameon=False)
fig.suptitle(f"Jednoprzedziałowy surogat ODE_1 ($K_{{eff}}={K_EFF_BASE:.0f}$, $c=c_0$) a pełne PDE", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_01_ode1.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Wnioski

- **Pole pokrywa guz ($c_0 \approx 1$):** `ODE_1` jakościowo i ilościowo dobrze oddaje spadek masy podczas frakcji — cały guz dostaje pełną dawkę, więc jednorodny współczynnik $c \approx 1$ jest bliski prawdy.
- **Pole za wąskie ($c_0 \approx 0{,}49$) i pole przesunięte ($c_0 \approx 0{,}67$):** pojedyncza zmienna $M(t)$ **przeszacowuje efekt leczenia** w trakcie kolejnych frakcji i zaniża minimum masy oraz tempo odrostu. Powód jest strukturalny: w PDE pierwsza frakcja zabija komórki *wewnątrz* wiązki, więc w kolejnych frakcjach niemal cała pozostała masa leży już *poza* wiązką i kolejne frakcje przestają ją zabijać ($c_{\text{efektywne}}(t) \to 0$). W `ODE_1` ułamek $c$ pozostaje stały, więc każda frakcja nadal redukuje masę o $e^{-\beta c}$.
- Rozwiązaniem tego ograniczenia jest model dwuprzedziałowy `ODE_2` (osobno masa w wiązce i poza wiązką — issue #19).